# 09 · Recomendación: dónde abrir, qué tipo, cuánto vendería y cuánto ganaría

**Objetivo:** para cada zona (hexágono) y tipo de restaurante, estimar:

| Qué | Cómo |
|---|---|
| **Probabilidad de seguir abierto a 5 años** | Modelo logístico entrenado con lo que pasó entre 2020 y 2025, ahora con variables de dinero (demanda, oferta, saturación) |
| **Ventas anuales** | Dos alternativas: (1) **productividad**: ventas por trabajador de su tipo y municipio (Censos Económicos) × tamaño típico; (2) **cuota de mercado**: parte del gasto en comer fuera del entorno que le tocaría según su tamaño frente a los restaurantes vecinos (idea del modelo de Huff), con un tope de capacidad |
| **Utilidad anual** | Ventas × margen de su tipo y municipio |
| **Valor esperado a 5 años** | Utilidad anual sumada año por año, multiplicada por la probabilidad de seguir abierto cada año |

Con eso se ordenan las combinaciones zona × tipo. El valor esperado es un **índice para comparar**, no una promesa de ganancia: las ventas y la utilidad son promedios estimados, no datos de un local real.

**Cómo se valida:** el modelo de permanencia se evalúa **fuera de muestra**, dejando fuera zonas completas (validación espacial), y el índice de 2020 se compara con lo que pasó en 2025.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import statsmodels.formula.api as smf
from scipy import stats
import h3
import mapas

ROOT = Path.cwd().parent
PROC, FIG = ROOT / "data/processed", ROOT / "figures"
RES = 8

## 1. Configuración

- `HORIZONTE`: años del valor esperado.
- `SUPERVIVENCIA_NUEVOS`: el modelo se entrena con restaurantes de 2020 de todas las edades, que sobreviven más que uno recién abierto. Para un local **nuevo** se reescala la probabilidad para que, en promedio, coincida con la supervivencia a 5 años de negocios recién nacidos del simulador del INEGI (Querétaro, servicios privados no financieros: 0.387).
- `MIN_POBLACION`, `MIN_EMPLEO`: una zona entra a la recomendación si tiene gente que viva o trabaje ahí.

In [ ]:
HORIZONTE = 5
SUPERVIVENCIA_NUEVOS = 0.387
MIN_POBLACION, MIN_EMPLEO = 500, 300
FOLDS, SEMILLA = 5, 42
MUN_CVE = {"Corregidora": 6, "Huimilpan": 8, "El Marqués": 11, "Querétaro": 14}

## 2. Datos

In [ ]:
trans = pd.read_parquet(PROC / "transiciones_restaurantes_2020_2025.parquet")
ing20 = pd.read_parquet(PROC / "restaurantes_ingresos_2020.parquet")
ing25 = pd.read_parquet(PROC / "restaurantes_ingresos_2025.parquet")
ent = pd.read_parquet(PROC / "entradas_restaurantes_2020_2025.parquet")
oferta = pd.read_parquet(PROC / f"oferta_hex{RES}.parquet").set_index("h3")
dem = pd.read_parquet(PROC / f"demanda_hex{RES}.parquet").set_index("h3")
base = pd.read_parquet(PROC / f"hex_res{RES}_base_2020.parquet").set_index("h3")
param = pd.read_parquet(PROC / "ce2024_parametros_mun_tipo.parquet")
cal = pd.read_parquet(PROC / "ce2024_calibracion_mun.parquet").set_index("cve_mun")["factor"]
TIPOS = sorted(trans["tipo"].unique())

hexes = dem.join(oferta, how="outer").fillna(0).join(base[["POBTOT", "escolaridad"]], how="left")
hexes["escolaridad"] = hexes["escolaridad"].fillna(base["escolaridad"].median())
hexes["POBTOT"] = hexes["POBTOT"].fillna(0)

# Municipio de cada hexágono: el más común entre sus establecimientos de 2025
den25 = pd.read_parquet(PROC / "denue_zmq_2025-05.parquet", columns=["latitud", "longitud", "mun_nombre"])
den25["h3"] = [h3.latlng_to_cell(la, lo, RES) for la, lo in zip(den25["latitud"], den25["longitud"])]
hexes["municipio"] = den25.groupby("h3")["mun_nombre"].agg(lambda s: s.mode().iat[0])
print(f"Hexágonos con datos: {len(hexes):,}")

## 3. Variables de dinero para cada restaurante de 2020

Todas medidas en 2020, antes de saber si el restaurante siguió:

| Variable | Qué mide |
|---|---|
| `tamano` | log del personal estimado del propio restaurante |
| `demanda` | log del gasto anual en comer fuera en el entorno |
| `saturacion` | log(oferta del entorno / demanda del entorno): >0, más ventas que gasto local |
| `competencia_tipo` | parte de la oferta del entorno que es del mismo tipo |
| `escolaridad` | escolaridad promedio del hexágono (nivel socioeconómico) |

In [ ]:
def variables(df, anio, ventas_propias=None):
    h = hexes.reindex(df["h3"])
    dem_e = h[f"demanda_{anio}_entorno"].to_numpy()
    of_e = h[f"oferta_{anio}_entorno"].to_numpy() + (0 if ventas_propias is None else ventas_propias)
    of_tipo = np.array([hexes.at[c, f"oferta_{anio}__{t}_entorno"] if c in hexes.index and f"oferta_{anio}__{t}_entorno" in hexes.columns else 0
                        for c, t in zip(df["h3"], df["tipo"])]) + (0 if ventas_propias is None else ventas_propias)
    out = pd.DataFrame(index=df.index)
    out["demanda"] = np.log1p(dem_e)
    out["saturacion"] = np.log((of_e + 1) / (dem_e + 1))
    out["competencia_tipo"] = np.where(of_e > 0, of_tipo / np.maximum(of_e, 1), 0)
    out["escolaridad"] = h["escolaridad"].to_numpy()
    return out

datos = trans[["id", "h3", "tipo", "mun_nombre", "continua"]].merge(ing20[["id", "empleo", "ventas_est"]], on="id", how="left")
datos["tamano"] = np.log(datos["empleo"].fillna(3))
datos = datos.join(variables(datos, "2020"))
datos["continua"] = datos["continua"].astype(int)
VARS_DINERO = ["tamano", "demanda", "saturacion", "competencia_tipo", "escolaridad"]
medias, desv = datos[VARS_DINERO].mean(), datos[VARS_DINERO].std()
Z = lambda d: (d[VARS_DINERO] - medias) / desv           # misma estandarización para entrenar y para recomendar
datos[[f"z_{v}" for v in VARS_DINERO]] = Z(datos).to_numpy()
datos = datos.dropna(subset=[f"z_{v}" for v in VARS_DINERO])
print(f"Restaurantes para entrenar: {len(datos):,}")
datos[VARS_DINERO].describe().round(2)

## 4. Modelo de permanencia con dinero

Se comparan dos modelos logísticos (errores agrupados por hexágono):

- **Sin dinero:** tipo de restaurante, municipio y escolaridad.
- **Con dinero:** lo anterior más tamaño, demanda, saturación y competencia del mismo tipo.

In [ ]:
Zs = [f"z_{v}" for v in VARS_DINERO]
f_sin = "continua ~ C(tipo) + C(mun_nombre) + z_escolaridad"
f_con = "continua ~ C(tipo) + C(mun_nombre) + " + " + ".join(Zs)
grupos = pd.factorize(datos["h3"])[0]
m_sin = smf.logit(f_sin, datos).fit(disp=0, cov_type="cluster", cov_kwds={"groups": grupos})
m_con = smf.logit(f_con, datos).fit(disp=0, cov_type="cluster", cov_kwds={"groups": grupos})
print(f"Pseudo R² → sin dinero: {m_sin.prsquared:.4f} · con dinero: {m_con.prsquared:.4f}")
print(f"AIC → sin dinero: {m_sin.aic:,.1f} · con dinero: {m_con.aic:,.1f} (menor es mejor)")
ci = m_con.conf_int()
efectos = pd.DataFrame({"razón de momios": np.exp(m_con.params[Zs]), "inf_95": np.exp(ci.loc[Zs, 0]),
                        "sup_95": np.exp(ci.loc[Zs, 1]), "p": m_con.pvalues[Zs]})
efectos.index = VARS_DINERO
efectos.round(3)

## 5. ¿Qué tan bien pronostica? Validación espacial fuera de muestra

Los hexágonos se agrupan en zonas grandes (H3 resolución 6, unos 36 km²) y se reparten en 5 grupos. Cada vez se entrena con 4 grupos y se pronostica el quinto, así el modelo nunca ve la zona que pronostica. La métrica es el **AUC**: la probabilidad de que el modelo le dé más probabilidad de seguir a un restaurante que siguió que a uno que salió (0.5 = azar, 1 = perfecto).

In [ ]:
def auc(y, p):
    r = stats.rankdata(p)
    n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

zonas = datos["h3"].map(lambda c: h3.cell_to_parent(c, 6))
orden = pd.Series(zonas.unique()).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
fold_de = {z: i % FOLDS for i, z in enumerate(orden)}
datos["fold"] = zonas.map(fold_de)

oof = pd.DataFrame(index=datos.index, columns=["sin", "con"], dtype=float)
for k in range(FOLDS):
    tr, te = datos[datos["fold"] != k], datos[datos["fold"] == k]
    for nombre, f in (("sin", f_sin), ("con", f_con)):
        m = smf.logit(f, tr).fit(disp=0)
        te_ok = te[te["tipo"].isin(tr["tipo"].unique()) & te["mun_nombre"].isin(tr["mun_nombre"].unique())]
        oof.loc[te_ok.index, nombre] = m.predict(te_ok)
ok = oof.notna().all(axis=1)
y = datos.loc[ok, "continua"].to_numpy()
print(f"AUC fuera de muestra → sin dinero: {auc(y, oof.loc[ok, 'sin'].to_numpy()):.3f} · con dinero: {auc(y, oof.loc[ok, 'con'].to_numpy()):.3f}")
datos["p_oof"] = oof["con"]

cal_tab = (datos.loc[ok].assign(decil=pd.qcut(datos.loc[ok, "p_oof"], 10, labels=False, duplicates="drop"))
           .groupby("decil").agg(predicha=("p_oof", "mean"), observada=("continua", "mean"), n=("continua", "size")))
cal_tab.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], ls="--", color="#999", lw=1)
ax.plot(cal_tab["predicha"], cal_tab["observada"], "o-", color="#14213D")
ax.set_xlabel("Probabilidad predicha (fuera de muestra)"); ax.set_ylabel("Proporción que siguió")
ax.set_xlim(0.2, 0.9); ax.set_ylim(0.2, 0.9); ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Calibración del pronóstico de permanencia, por decil", loc="left")
plt.tight_layout(); plt.savefig(FIG / "09_calibracion.png", dpi=200, bbox_inches="tight"); plt.show()

## 6. ¿La zona cambia cuánto vende un restaurante?

Los Censos Económicos dan ventas por trabajador por municipio, no por zona. Para saber si una zona con más demanda permite vender más, se usa el **tamaño**: si en zonas con más demanda por cada peso de oferta los restaurantes son más grandes, venden más. Se estima la elasticidad θ:

log(personal) = tipo + municipio + θ · log(demanda del entorno / oferta del entorno sin el propio restaurante)

Si θ no es significativo o es negativo, se usa θ = 0 (sin ajuste por zona).

In [ ]:
r = ing25.copy()
h = hexes.reindex(r["h3"])
of_sin = (h["oferta_2025_entorno"].to_numpy() - r["ventas_est"].to_numpy()).clip(min=1)
r["log_razon"] = np.log((h["demanda_2025_entorno"].to_numpy() + 1) / of_sin)
r["log_empleo"] = np.log(r["empleo"])
r = r.replace([np.inf, -np.inf], np.nan).dropna(subset=["log_razon", "log_empleo", "cve_mun"])
m_tam = smf.ols("log_empleo ~ C(tipo) + C(cve_mun) + log_razon", r).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(r["h3"])[0]})
theta, p_theta = m_tam.params["log_razon"], m_tam.pvalues["log_razon"]
THETA = theta if (theta > 0 and p_theta < 0.05) else 0.0
print(f"θ estimado = {theta:.3f} (p = {p_theta:.3f}) → θ usado = {THETA:.3f}")

## 7. La recomendación (condiciones de 2025)

Para cada hexágono con gente o empleo, y cada tipo de restaurante, se simula **abrir un local nuevo de tamaño típico** (la mediana de personal de los restaurantes de ese tipo que entraron entre 2020 y 2025):

1. **Ventas base** = ventas por trabajador (tipo, municipio) × personal típico × calibración del municipio.
2. **Ajuste por zona** = (razón demanda/oferta del hexágono ÷ razón mediana)^θ, acotado entre 0.5 y 2.
3. **Probabilidad a 5 años** = modelo con dinero, reescalada a negocios nuevos.
4. **Utilidad anual** = ventas × margen.
5. **Valor esperado a 5 años** = Σ (utilidad × probabilidad de seguir abierto en el año t), con probabilidad anual constante.

In [ ]:
tam_tipico = (ing25[ing25["id"].isin(ent["id"])].groupby("tipo")["empleo"].median()
              .reindex(TIPOS).fillna(ing25.groupby("tipo")["empleo"].median()))
zonas_ok = hexes[((hexes["POBTOT"] >= MIN_POBLACION) | (hexes["empleo_2025"] >= MIN_EMPLEO)) & hexes["municipio"].notna()]
grid = pd.MultiIndex.from_product([zonas_ok.index, TIPOS], names=["h3", "tipo"]).to_frame(index=False)
grid["mun_nombre"] = grid["h3"].map(zonas_ok["municipio"])
grid["cve_mun"] = grid["mun_nombre"].map(MUN_CVE)
grid = grid.merge(param[["cve_mun", "tipo", "ventas_por_trab", "margen"]], on=["cve_mun", "tipo"], how="left")
grid["empleo"] = grid["tipo"].map(tam_tipico)
grid["ventas_base"] = grid["ventas_por_trab"] * grid["empleo"] * grid["cve_mun"].map(cal).fillna(1)

h = hexes.reindex(grid["h3"])
razon = (h["demanda_2025_entorno"].to_numpy() + 1) / (h["oferta_2025_entorno"].to_numpy() + grid["ventas_base"].to_numpy())
grid["ajuste_zona"] = np.clip((razon / np.nanmedian(razon)) ** THETA, 0.5, 2)
grid["ventas_anuales"] = grid["ventas_base"] * grid["ajuste_zona"]
grid["utilidad_anual"] = grid["ventas_anuales"] * grid["margen"]

grid["tamano"] = np.log(grid["empleo"])
grid = grid.join(variables(grid, "2025", ventas_propias=grid["ventas_anuales"].to_numpy()))
grid[Zs] = Z(grid).to_numpy()
p_modelo = m_con.predict(grid)
escala = SUPERVIVENCIA_NUEVOS / datos["continua"].mean()
grid["p_5_anios"] = np.clip(p_modelo * escala, 0.01, 0.99)
s_anual = grid["p_5_anios"] ** (1 / HORIZONTE)
grid["valor_5_anios"] = grid["utilidad_anual"] * sum(s_anual ** t for t in range(1, HORIZONTE + 1))
grid = grid.dropna(subset=["valor_5_anios"])
print(f"Combinaciones zona × tipo evaluadas: {len(grid):,} ({grid['h3'].nunique():,} hexágonos)")
grid[["ventas_anuales", "utilidad_anual", "p_5_anios", "valor_5_anios"]].describe().round(2)

### 7b. Alternativa 2: ventas por cuota de mercado

La alternativa 1 usa la productividad promedio del municipio, así que dentro de un municipio casi no distingue zonas. La alternativa 2 parte del dinero disponible en el entorno:

ventas = α × demanda del entorno × parte del tipo × cuota del local

- **α** = ventas totales de restaurantes / gasto total en comer fuera en la ZM: la parte del gasto que se queda en restaurantes.
- **Parte del tipo** = qué fracción de las ventas de restaurantes de la ZM es de ese tipo (por ejemplo, cuánto del gasto va a taquerías).
- **Cuota del local** = personal del local nuevo / (personal de los restaurantes **del mismo tipo** en el entorno + personal del local nuevo), con un tope de `CUOTA_MAX`: un solo local no se queda con más de esa parte del gasto de su tipo, aunque no tenga competencia cerca.
- **Tope de capacidad:** un local de tamaño típico no puede vender sin límite. Las ventas por cuota se acotan entre `CAP_MIN` y `CAP_MAX` veces las ventas base de la alternativa 1 (la productividad de su tipo y municipio). Sin este tope, en zonas con mucho gasto y poca competencia del mismo tipo, un local de 3 a 5 personas "vendería" decenas de millones al año.
- α puede ser **mayor que 1**: los restaurantes también venden a visitantes y turistas, y la ENIGH tiende a captar menos gasto del real.
- Supone que cada restaurante captura gasto en proporción a su tamaño y que la gente come en su entorno (hexágono + 6 vecinos). Es una versión simplificada del modelo de Huff, sin atracción ni distancia.

Las dos alternativas usan la misma probabilidad de seguir abierto.

In [ ]:
CUOTA_MAX = 0.25
CAP_MIN, CAP_MAX = 0.5, 2.0      # mismo rango que el ajuste por zona de la alternativa 1
emp_tipo = ing25.groupby(["h3", "tipo"])["empleo"].sum()
vecinos = {c: list(h3.grid_disk(c, 1)) for c in zonas_ok.index}
emp_tipo_ent = np.array([emp_tipo.reindex([(v, t) for v in vecinos[c]]).sum() for c, t in zip(grid["h3"], grid["tipo"])])
parte_tipo = ing25.groupby("tipo")["ventas_est"].sum() / ing25["ventas_est"].sum()
alpha = oferta["oferta_2025"].sum() / dem["demanda_2025"].sum()
cuota = np.minimum(CUOTA_MAX, grid["empleo"] / (emp_tipo_ent + grid["empleo"]))
ventas_mercado = (alpha * hexes.reindex(grid["h3"])["demanda_2025_entorno"].to_numpy()
                  * grid["tipo"].map(parte_tipo).to_numpy() * cuota)
grid["ventas_cuota"] = np.clip(ventas_mercado, CAP_MIN * grid["ventas_base"], CAP_MAX * grid["ventas_base"])
print(f"Combinaciones en el tope de capacidad: {(ventas_mercado >= CAP_MAX * grid['ventas_base']).mean():.0%} arriba · {(ventas_mercado <= CAP_MIN * grid['ventas_base']).mean():.0%} abajo")
grid["utilidad_cuota"] = grid["ventas_cuota"] * grid["margen"]
grid["valor_5_anios_cuota"] = grid["utilidad_cuota"] * sum(s_anual.loc[grid.index] ** t for t in range(1, HORIZONTE + 1))
rho = stats.spearmanr(grid["valor_5_anios"], grid["valor_5_anios_cuota"]).statistic
print(f"α (parte del gasto en comer fuera que va a restaurantes) = {alpha:.2f}")
print(f"Correlación de rangos entre las dos alternativas: ρ = {rho:.2f}")
grid[["ventas_anuales", "ventas_cuota", "utilidad_anual", "utilidad_cuota"]].describe().round(0)

## 8. Las mejores combinaciones

In [ ]:
print("Alternativa 1 · productividad")
top = grid.sort_values("valor_5_anios", ascending=False).head(15)
top[["h3", "mun_nombre", "tipo", "ventas_anuales", "utilidad_anual", "p_5_anios", "valor_5_anios"]].round(
    {"ventas_anuales": 0, "utilidad_anual": 0, "p_5_anios": 2, "valor_5_anios": 0})

In [ ]:
print("Alternativa 2 · cuota de mercado")
top2 = grid.sort_values("valor_5_anios_cuota", ascending=False).head(15)
top2[["h3", "mun_nombre", "tipo", "ventas_cuota", "utilidad_cuota", "p_5_anios", "valor_5_anios_cuota"]].round(
    {"ventas_cuota": 0, "utilidad_cuota": 0, "p_5_anios": 2, "valor_5_anios_cuota": 0})

In [ ]:
por_tipo = (grid.groupby("tipo").agg(ventas_mediana=("ventas_anuales", "median"), utilidad_mediana=("utilidad_anual", "median"),
                                     p_mediana=("p_5_anios", "median"), valor_mediano=("valor_5_anios", "median"),
                                     valor_mediano_cuota=("valor_5_anios_cuota", "median"))
            .sort_values("valor_mediano", ascending=False))
por_tipo.round(2)

## 9. Mapa · El tipo recomendado en cada zona, con cada alternativa

In [ ]:
PALETA = ["#14213D", "#2E6F95", "#8DB7D3", "#E8A25F", "#A8551A", "#6B8F71", "#C9C2B4", "#7A5C99", "#D1495B", "#3E7CB1"]
color_tipo = {t: PALETA[i % len(PALETA)] for i, t in enumerate(TIPOS)}
for col, nombre, archivo in (("valor_5_anios", "productividad", "09_mapa_tipo_recomendado.png"),
                             ("valor_5_anios_cuota", "cuota de mercado", "09_mapa_tipo_recomendado_cuota.png")):
    mejor_alt = grid.loc[grid.groupby("h3")[col].idxmax()].set_index("h3")
    fig, ax, polys = mapas.base_mapa(mejor_alt.index)
    mapas.capa_hex(ax, polys, facecolors=mejor_alt["tipo"].map(color_tipo).values)
    mapas.referencias(ax)
    presentes = mejor_alt["tipo"].value_counts()
    ax.legend(handles=[Patch(color=color_tipo[t], label=f"{t} ({n})") for t, n in presentes.items()],
              loc="lower left", fontsize=10, title="Tipo con mayor valor esperado (hexágonos)")
    ax.set_title(f"Tipo recomendado por zona · alternativa {nombre}", loc="left", fontsize=13)
    plt.tight_layout(); plt.savefig(FIG / archivo, dpi=200, bbox_inches="tight"); plt.show()
mejor = grid.loc[grid.groupby("h3")["valor_5_anios_cuota"].idxmax()].set_index("h3")

## 10. Mapa · Valor esperado del mejor tipo (cuota de mercado)

In [ ]:
v = mejor["valor_5_anios_cuota"] / 1e6
BORDES = list(np.round(np.quantile(v, [0, 0.2, 0.4, 0.6, 0.8]), 2)) + [v.max() + 0.01]
ETIQ = [f"{a:.2f}–{b:.2f}" for a, b in zip(BORDES[:-1], BORDES[1:])]
cmap, norm = mapas.escala_por_clases(BORDES, ["#EAF1F6", "#BFD6E6", "#8DB7D3", "#2E6F95", "#14213D"])
fig, ax, polys = mapas.base_mapa(v.index)
coll = mapas.capa_hex(ax, polys, array=v.values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "Valor esperado a 5 años (millones de pesos de 2023)")
ax.set_title("Valor esperado del tipo recomendado · por quintiles", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "09_mapa_valor_esperado.png", dpi=200, bbox_inches="tight"); plt.show()

## 11. Prueba hacia atrás: ¿el índice de 2020 anticipó lo que pasó?

Con información de 2020 (y probabilidades **fuera de muestra**), se calcula para cada hexágono × tipo con al menos 3 restaurantes:

- la probabilidad predicha promedio, comparada con la **permanencia observada** 2020 → 2025;
- el valor esperado, comparado con el **crecimiento neto observado** (entradas − salidas) / restaurantes de 2020.

Se reporta la correlación de rangos (Spearman). Positiva = el índice ordena bien las zonas.

In [ ]:
matriz = pd.read_parquet(PROC / f"matriz_hex{RES}_tipo_2020_2025.parquet")
celdas = matriz[matriz["n_base"] >= 3].copy()
celdas["crecimiento"] = celdas["neto"] / celdas["n_base"]
pred = datos.groupby(["h3", "tipo"]).agg(p_pred=("p_oof", "mean"), utilidad=("ventas_est", "median"))
mg = param.set_index(["cve_mun", "tipo"])["margen"]
celdas = celdas.merge(pred, left_on=["h3", "tipo"], right_index=True, how="inner").dropna(subset=["p_pred"])
celdas["cve_mun"] = celdas["h3"].map(hexes["municipio"]).map(MUN_CVE)
celdas["utilidad"] = celdas["utilidad"] * [mg.get((m, t), np.nan) for m, t in zip(celdas["cve_mun"], celdas["tipo"])]
s = celdas["p_pred"] ** (1 / HORIZONTE)
celdas["valor"] = celdas["utilidad"] * sum(s ** t for t in range(1, HORIZONTE + 1))
r1 = stats.spearmanr(celdas["p_pred"], celdas["permanencia"], nan_policy="omit")
r2 = stats.spearmanr(celdas["valor"], celdas["crecimiento"], nan_policy="omit")
print(f"Celdas zona × tipo: {len(celdas):,}")
print(f"Probabilidad predicha vs permanencia observada: ρ = {r1.statistic:.2f} (p = {r1.pvalue:.3f})")
print(f"Valor esperado vs crecimiento neto observado: ρ = {r2.statistic:.2f} (p = {r2.pvalue:.3f})")

## 12. Guardar

In [ ]:
cols = ["h3", "mun_nombre", "tipo", "empleo", "ventas_anuales", "utilidad_anual", "ajuste_zona", "ventas_cuota", "utilidad_cuota", "p_5_anios", "valor_5_anios", "valor_5_anios_cuota"]
grid[cols].to_parquet(PROC / "recomendaciones_hex_tipo.parquet", index=False)
grid[cols].sort_values("valor_5_anios", ascending=False).to_csv(PROC / "recomendaciones_hex_tipo.csv", index=False, encoding="utf-8-sig")
efectos.to_csv(PROC / "modelo_permanencia_dinero_efectos.csv", encoding="utf-8-sig")
print("Guardado: recomendaciones_hex_tipo (parquet y csv) y efectos del modelo")

## 13. Cómo leer y cómo no leer esta recomendación

- **Las ventas y la utilidad son estimaciones promedio.** La alternativa 1 casi no distingue zonas dentro de un municipio; la 2 sí, pero supone que cada restaurante captura gasto según su tamaño y que la gente come cerca. Ninguna incluye precio, calidad, renta del local ni inversión inicial. Donde las dos coinciden, la recomendación es más sólida.
- **La probabilidad viene de un solo periodo** (2020–2025, con pandemia) y de asociaciones, no causas.
- **La demanda usa la población de 2020** y el empleo de 2025; la periferia que creció después de 2020 queda subestimada.
- **El gasto de la ENIGH es por hogar y se lleva a zonas con promedios**: es una aproximación de pequeñas áreas, no un dato medido en cada hexágono.
- **Pesos de años distintos:** las ventas están en pesos de 2023 (Censos Económicos) y la demanda en pesos de 2024 (ENIGH). Para comparar zonas no importa; para montos absolutos, sí.
- **Úsalo para ordenar opciones**, no para decidir una inversión sin visitar la zona.